# Entity Extraction Approach

A rule-based entity extraction pipeline was developed to extract structured information from unstructured Indian bank transaction narrations. The pipeline identifies three key components from each narration:

- **Transaction Channel** – What payment channel or mode was used?
- **Counterparty** – Who was the actual merchant, person, or institution involved in the transaction?
- **Entity Role** – Was the counterparty sending or receiving money?

---

## Phase 1 – Text Normalization

Transaction narrations are first normalized to improve parsing consistency. Common spacing inconsistencies (e.g., `PH ONEPE`, `PHONE PE`, `PAY TM`, `GOOGLE PAY`) and noisy delimiters (such as pipe `|` characters) are standardized so that payment gateway detection remains reliable across different banks and narration formats.

---

## Phase 2 – Payment Channel Detection

The parser first determines whether a transaction is UPI-based. If so, it identifies the corresponding payment gateway whenever possible, including:

- PhonePe
- Paytm
- Google Pay
- Razorpay
- Airtel Payments
- BharatPe
- Amazon Pay
- SBIePay

If the transaction is not UPI, the parser classifies it into the appropriate banking channel, such as:

- IMPS
- NEFT
- RTGS
- ACH/NACH
- ATM
- Cash
- POS/Card
- Loan Recovery
- Bank Charges
- Interest
- Fund Transfer

---

## Phase 3 – Counterparty Extraction (UPI)

For UPI transactions, the parser extracts the most meaningful counterparty from the Virtual Payment Address (VPA). Typically, the portion before the `@` symbol represents the merchant or individual.

**Example:**

```
MYNTRA@ybl  →  Myntra
```

However, QR-generated handles such as:

```
paytmqr...@paytm
Q899838377@ybl
```

are not treated as valid counterparties because they represent QR identifiers rather than meaningful merchant or person names. These are classified as `UNRESOLVED_QR_MERCHANT`.

---

## Phase 4 – Counterparty Extraction (Non-UPI)

For non-UPI transactions—including IMPS, NEFT, IFN, ACH, POS/Card, ATM, Cash, and Loan Recovery narrations—the parser applies format-specific extraction rules to identify the most relevant merchant, institution, or individual mentioned in the narration. Each channel type has its own dedicated extractor that understands the narration structure specific to that payment mode.

---

## Phase 5 – UPI Handle-to-Bank Resolution

When the actual counterparty name cannot be extracted (e.g., due to masking or missing data), the parser resolves the UPI banking handle from the `@` suffix into a human-readable bank name using a comprehensive dictionary of 60+ Indian UPI handle mappings.

**Examples:**

| UPI Handle | Resolved Bank |
|---|---|
| `@ybl` | YES Bank |
| `@okhdfcbank` | HDFC Bank |
| `@ibl` | IndusInd Bank |
| `@paytm` | Paytm Payments Bank |
| `@ptyes` | YES Bank |
| `@superyes` | YES Bank |
| `@fdrl` | Federal Bank |
| `@cbin` | Central Bank of India |

The handle extraction regex is designed to stop at delimiter characters (dashes, dots), ensuring that routing metadata embedded after the handle (e.g., `@PTYES-BARB0KHARAG-689729445387-SENT`) does not contaminate the bank name resolution.

Coverage includes all major Indian banks (HDFC, ICICI, SBI, Axis, YES Bank, IndusInd, Kotak, PNB, Canara, Union, etc.), small finance and payments banks (Equitas, Ujjivan, AU, Fino, Jio, NSDL, India Post), and fintech wallets (Freecharge, MobiKwik, Slice, Jupiter, CRED).

---

## Phase 6 – Compound Word Splitting

Indian bank narrations frequently concatenate multiple tokens into a single word with no spaces—particularly when gateway names, bank names, and corporate suffixes are merged together. The parser applies a compound word splitting step before token-level noise removal, inserting spaces around recognized substrings so they can be individually identified and stripped.

**Examples:**

| Raw Concatenated Text | After Splitting | After Noise Removal |
|---|---|---|
| `Phonepeprivatel` | `Phonepe privatel` | _(both stripped → UNKNOWN)_ |
| `Playstoreaxisbank` | `Playstore axisbank` | _(both stripped → UNKNOWN)_ |
| `Jiocitibank` | `Jio citibank` | _(both stripped → UNKNOWN)_ |
| `Bhartiairtelltd` | `Bharti airtel ltd` | _(all stripped → UNKNOWN)_ |

This step ensures that gateway and bank names embedded inside compound words do not leak through as false counterparties.

---

## Phase 7 – Noise Removal and Edge-Case Handling

A comprehensive noise removal system operates at multiple levels:

### Token-Level Noise Removal

Over 120 noise tokens are stripped during extraction, organized into categories:

- **Transaction keywords:** `UPI`, `UPIOUT`, `PAYMENT`, `FROM`, `ACCOUNT`, `NA`, `SENT`, `PAID`, `ONLINE`
- **Payment modes:** `ACH`, `NACH`, `IMPS`, `NEFT`, `RTGS`
- **Corporate suffixes (including truncated forms):** `LIMITED`, `LTD`, `PRIVATE`, `PVT`, `LIMITE`, `LIMIT`, `PRIVATEL`
- **Bank names and acronyms:** `HDFC`, `ICICI`, `SBI`, `AXIS`, `YES`, `YESB`, `KOTAK`, `PNB`, `CANARA`, `INDUSIND`, and 40+ others
- **Payment gateways:** `PHONEPE`, `PAYTM`, `GOOGLEPAY`, `RAZORPAY`, `BHARATPE`, `AMAZONPAY`, `BILLDESK`, `CASHFREE`
- **Settlement and routing junk:** `ESCROW`, `PAYOUT`, `STLMNT`, `AGGR`, `AGG`, `SETTL`, `NOD`, `NODA`, `ACCO`, `MERCHANT`

### Pattern-Based Noise Removal

Regex patterns catch technical artifacts that cannot be enumerated as fixed tokens:

- **IFSC codes** (e.g., `HDFC0001234`)
- **Bank routing codes** (e.g., `UTIB00512`)
- **QR identifiers** (e.g., `PAYTMQR2847561`, `Q899838377`)
- **IMPS routing codes** (e.g., `F09`, `F10`)
- **Long alphanumeric reference IDs** (10+ mixed characters)
- **Masked mobile numbers** (e.g., `MOBILEXXXXX`)

### Edge-Case Classification

Instead of assigning all unresolved cases to a generic `UNKNOWN` category, the parser classifies common edge cases into more informative categories:

**Masked Privacy Strings**

Narrations containing heavily redacted identifiers (e.g., `MOBILEXXXX`) are classified with contextual metadata:

```
MASKED_COUNTERPARTY (Merchant) - HDFC Bank
MASKED_COUNTERPARTY (Individual) - YES Bank
MASKED_COUNTERPARTY - IndusInd Bank
```

The classification includes:
- **Entity type** when available: `(Merchant)` for P2M transactions, `(Individual)` for P2P/P2A transactions
- **Resolved bank name** from the UPI handle

**Unresolved QR Merchants**

QR-code-generated payment handles that cannot be traced to a specific merchant are classified as:

```
UNRESOLVED_QR_MERCHANT
```

**Transaction References**

Long mixed alphanumeric identifiers commonly found in ACH/NACH transactions or backend banking systems are classified as:

```
TRANSACTION_REFERENCE
```

**Unknown with Context**

When no counterparty can be extracted but transaction metadata is available, the parser provides context-enriched unknown labels:

```
UNKNOWN_MERCHANT (P2M) - Axis Bank
UNKNOWN_INDIVIDUAL (P2P) - HDFC Bank
```

**Gibberish Filtering**

Random character sequences with little linguistic meaning (e.g., `Zxzy`, `Zvm`) are detected by checking for the absence of vowels and rejected rather than being treated as valid counterparties.

---

## Phase 8 – Entity Role Classification

The counterparty's role is inferred from the transaction type:

| Transaction Type | Entity Role |
|---|---|
| CREDIT | Payer (Money Source) |
| DEBIT | Payee (Money Recipient) |

---

## Phase 9 – Audit Metadata

To improve transparency and facilitate manual review, the parser generates additional audit fields for every extraction, including:

- **Confidence Score** – Indicates the reliability of the extracted entity, ranging from 0.0 (no candidate found) to 0.99 (high-confidence masked entity detection). Different extraction methods carry different confidence levels:

| Confidence | Extraction Method |
|---|---|
| 0.99 | Masked entity detection |
| 0.86 | UPI VPA or name extraction |
| 0.88 | Loan recipient extraction |
| 0.82 | IMPS/IFN name extraction |
| 0.80 | NEFT named segment |
| 0.78 | ACH/Card merchant |
| 0.72 | Cash/ATM location |
| 0.64 | Best delimited segment |
| 0.60 | QR code detection |
| 0.45 | Fallback cleaned narration |
| 0.40 | Fallback P2M/P2P/reference |
| 0.00 | No candidate found |

- **Extraction Rule** – Records the specific rule or pattern responsible for the extraction (e.g., `upi_vpa_or_name`, `masked_mobile_rule`, `neft_named_segment`, `fallback_p2m`).

These audit fields make it easier to identify low-confidence extractions and review fallback cases separately.


In [ ]:
import pandas as pd
from google.colab import files

uploaded = files.upload()

csv_file = list(uploaded.keys())[0]
df = pd.read_csv(csv_file)

df = df.dropna(subset=["NARRATION"]).copy()
df.head()

Saving Banking Sample Data for Entity Detection.csv to Banking Sample Data for Entity Detection (1).csv


,NARRATION,AAID,TYPE,MODE,AMOUNT,CURRENTBALANCE,VALUEDATE
0,UPIOUT/415442131843/Q899838377@ybl/Payment f/5411,aa91777914703824768192900488347968430422,DEBIT,OTHERS,70,7530,00:00.0
1,UPIOUT/415958720613/paytmqre4lk0uevrg@paytm//5451,aa91777914703824768192900488347968430422,DEBIT,OTHERS,60,1649,00:00.0
2,UPIOUT/416160374009/paytmqr14clec@paytm/Paym/5812,aa91777914703824768192900488347968430422,DEBIT,OTHERS,36,8730,00:00.0
3,UPIOUT/415838737884/Q047024853@ybl/Payment f/5812,aa91777914703824768192900488347968430422,DEBIT,OTHERS,5,8019,00:00.0
4,UPIOUT/416102104755/paytmqr281005050101wntva/5411,aa91777914703824768192900488347968430422,DEBIT,OTHERS,24,11040,00:00.0


In [ ]:
from __future__ import annotations

import re
from dataclasses import asdict, dataclass
from typing import Mapping

# ═════════════════════════════════════════════════════════════════════════════
# Constants
# ═════════════════════════════════════════════════════════════════════════════

UNKNOWN = "UNKNOWN"
MASKED_COUNTERPARTY = "MASKED_COUNTERPARTY"
TRANSACTION_REFERENCE = "TRANSACTION_REFERENCE"


@dataclass(frozen=True)
class ExtractionResult:
    narration: str
    payment_mode: str
    counterparty: str
    flow: str
    confidence: float
    rule: str

    def to_dict(self) -> dict[str, str | float]:
        return asdict(self)


# ═════════════════════════════════════════════════════════════════════════════
# Phase 2 – Payment Channel Detection
#
# Determines whether a transaction is UPI-based and identifies the
# corresponding payment gateway. If not UPI, classifies into the
# appropriate banking channel (IMPS, NEFT, RTGS, ACH/NACH, ATM, etc.).
# ═════════════════════════════════════════════════════════════════════════════

GATEWAY_PATTERNS: tuple[tuple[str, re.Pattern[str]], ...] = (
    ("PhonePe",         re.compile(r"\bPH\s*ONE\s*PE\b|\bPHONE\s*PE\b|\bPHONEPE\b|@YBL\b|@AXL\b", re.I)),
    ("Paytm",           re.compile(r"\bPAY\s*TM\b|\bPAYTM\b|@PAYTM\b|PAYTMQR", re.I)),
    ("Google Pay",      re.compile(r"\bGOOGLE\s*PAY\b|\bGPAY\b|@OK(?:AXIS|AXI|HDFC|ICICI|SBI)\b", re.I)),
    ("Razorpay",        re.compile(r"\bRAZOR\s*PAY\b|\bRAZORPAY\b|\.RZP@|@RZP|RZP@", re.I)),
    ("Airtel Payments", re.compile(r"\bAIRTEL\b|@MAIRTEL\b|RXAIRTEL", re.I)),
    ("BharatPe",        re.compile(r"\bBHARAT\s*PE\b|\bBHARATPE\b", re.I)),
    ("Amazon Pay",      re.compile(r"\bAMAZON\s*PAY\b|\bAMAZONPAY\b", re.I)),
    ("SBIePay",         re.compile(r"\bSBI\s*E\s*PAY\b|\bSBIEPAY\b", re.I)),
)

MODE_PATTERNS: tuple[tuple[str, re.Pattern[str]], ...] = (
    ("ATM",           re.compile(r"\b(?:ATM|NFS|ATW)\b|ATM CASH|CASH TXN", re.I)),
    ("Cash",          re.compile(r"\bCASH\b|CASH DEPOSIT|CASH WDL", re.I)),
    ("IMPS",          re.compile(r"\bIMPS\b|\.IMPS", re.I)),
    ("NEFT",          re.compile(r"\bNEFT\b", re.I)),
    ("RTGS",          re.compile(r"\bRTGS\b", re.I)),
    ("ACH/NACH",      re.compile(r"\b(?:ACH|ACHCR|NACH|ECS)\b", re.I)),
    ("POS/Card",      re.compile(r"\b(?:POS|PRCR|CARD|VISA|MASTER(?:CARD)?)\b", re.I)),
    ("UPI",           re.compile(r"\bUPI(?:OUT|AB|AR)?\b|UPI\s+IN|@[A-Z0-9._-]+", re.I)),
    ("Loan Recovery", re.compile(r"\bLOAN\s+REC\b|\bLOAN\b", re.I)),
    ("Charges",       re.compile(r"\bCHARGE\b|\bCHARGES\b|\bCHG\b", re.I)),
    ("Interest",      re.compile(r"\bINTEREST\b|INT\.? CREDIT", re.I)),
    ("Fund Transfer", re.compile(r"\b(?:FT|TRTR|MB|MBK|INET|INF|IFN|MMT|RECD)\b", re.I)),
)


# ═════════════════════════════════════════════════════════════════════════════
# Phase 5 – UPI Handle-to-Bank Resolution
#
# When the actual counterparty name cannot be extracted (e.g., due to
# masking or missing data), the parser resolves the UPI banking handle
# from the @ suffix into a human-readable bank name using a comprehensive
# dictionary of 60+ Indian UPI handle mappings.
#
# The handle extraction regex stops at delimiter characters (dashes, dots),
# ensuring that routing metadata embedded after the handle does not
# contaminate the bank name resolution.
#
# Example:
#   @PTYES-BARB0KHARAG-689729445387  →  captures "PTYES"  →  "YES Bank"
#   @ okhdfcbank                      →  captures "okhdfcbank"  →  "HDFC Bank"
# ═════════════════════════════════════════════════════════════════════════════

UPI_HANDLE_BANK_MAP = {
    # HDFC Bank
    "okhdfcbank": "HDFC Bank", "hdfcbank": "HDFC Bank", "hdfc": "HDFC Bank",

    # ICICI Bank
    "okicici": "ICICI Bank", "icici": "ICICI Bank",

    # State Bank of India
    "oksbi": "State Bank of India", "sbi": "State Bank of India",
    "sbin": "State Bank of India",

    # Axis Bank
    "okaxis": "Axis Bank", "axisbank": "Axis Bank", "axl": "Axis Bank",
    "utib": "Axis Bank",

    # YES Bank (all routing variants)
    "ybl": "YES Bank", "yesbank": "YES Bank", "yesbankltd": "YES Bank",
    "yes": "YES Bank", "superyes": "YES Bank", "ptyes": "YES Bank",
    "yescred": "YES Bank", "yespop": "YES Bank", "yesc": "YES Bank",
    "nyes": "YES Bank", "yesba": "YES Bank", "yespay": "YES Bank",

    # IndusInd Bank
    "ibl": "IndusInd Bank", "indus": "IndusInd Bank",

    # Kotak Mahindra Bank
    "kotak": "Kotak Mahindra Bank", "kkbk": "Kotak Mahindra Bank",

    # Paytm Payments Bank
    "paytm": "Paytm Payments Bank",

    # Airtel Payments Bank
    "apl": "Airtel Payments Bank", "apb": "Airtel Payments Bank",
    "mairtel": "Airtel Payments Bank", "airtel": "Airtel Payments Bank",
    "airp": "Airtel Payments Bank",

    # IDFC First Bank
    "idfc": "IDFC First Bank", "idfcbank": "IDFC First Bank",

    # Federal Bank
    "fbl": "Federal Bank", "fdrl": "Federal Bank",

    # Bank of Baroda
    "barodampay": "Bank of Baroda", "barb": "Bank of Baroda",
    "bob": "Bank of Baroda",

    # RBL Bank
    "rbl": "RBL Bank", "ratn": "RBL Bank",

    # Central Bank of India
    "cbin": "Central Bank of India",

    # Union Bank of India
    "ubi": "Union Bank of India", "ubin": "Union Bank of India",
    "unionbank": "Union Bank of India",

    # Punjab National Bank
    "pnb": "Punjab National Bank", "punb": "Punjab National Bank",

    # Canara Bank
    "cnrb": "Canara Bank", "canara": "Canara Bank",

    # Indian Bank
    "indianbk": "Indian Bank", "idib": "Indian Bank",

    # Bank of India
    "bkid": "Bank of India", "boi": "Bank of India",

    # Indian Overseas Bank
    "ioba": "Indian Overseas Bank",

    # Karnataka Bank
    "karb": "Karnataka Bank",

    # South Indian Bank
    "sibl": "South Indian Bank",

    # Bandhan Bank
    "bandhan": "Bandhan Bank", "bdbl": "Bandhan Bank",

    # IDBI Bank
    "idbi": "IDBI Bank", "ibkl": "IDBI Bank",

    # UCO Bank
    "ucba": "UCO Bank", "uco": "UCO Bank",

    # Bank of Maharashtra
    "mahb": "Bank of Maharashtra",

    # Punjab & Sind Bank
    "psib": "Punjab & Sind Bank",

    # Citibank
    "citi": "Citibank", "citibank": "Citibank",

    # Small Finance & Payments Banks
    "equitas": "Equitas Small Finance Bank",
    "ujjivan": "Ujjivan Small Finance Bank",
    "aubank": "AU Small Finance Bank",
    "fino": "Fino Payments Bank",
    "jio": "Jio Payments Bank",
    "nsdl": "NSDL Payments Bank",
    "ipos": "India Post Payments Bank",

    # Payment Wallets & Fintechs
    "freecharge": "Freecharge",
    "mobikwik": "MobiKwik",
    "slice": "Slice",
    "jupiter": "Jupiter",
    "cred": "CRED",

    # Cooperative / Regional
    "sibi": "State Bank of India (Intl)",
}


def _resolve_bank_from_handle(raw_text: str) -> str:
    """
    Phase 5 – UPI Handle-to-Bank Resolution

    Finds the first UPI handle after @ (alphanumeric only — stops at dashes,
    dots, and other delimiters) and translates it to the actual bank name.
    """
    handle_match = re.search(r"@([A-Za-z0-9]+)", raw_text, re.I)
    if not handle_match:
        return ""
    raw_handle = handle_match.group(1).lower()
    return UPI_HANDLE_BANK_MAP.get(raw_handle, f"@{raw_handle}")


# ═════════════════════════════════════════════════════════════════════════════
# Phase 7 – Noise Removal and Edge-Case Handling
#
# Token-Level Noise Words: Over 120 noise tokens stripped during extraction,
# organized into categories — transaction keywords, payment modes, corporate
# suffixes (including truncated forms), bank names and acronyms, payment
# gateways, and settlement/routing junk.
# ═════════════════════════════════════════════════════════════════════════════

NOISE_WORDS = {
    # ── Basic Transaction Noise ──
    "", "PAY", "PAYM", "PAYMENT", "PAYMENTS", "PAYVIA", "FROM", "FOR", "REF",
    "TRANSFER", "FUND", "FUNDS", "IFN", "IN", "PRCR", "UPI", "UPIOUT", "UPIIN",
    "UPIAR", "UPIAB", "CR", "DR", "DEBIT", "CREDIT", "OUTWARD", "INWARD",
    "ACCOUNT", "MOBILEXXXX", "MOBILEXXXXX", "NA", "SENT", "PAID", "RECEIVED",
    "TXN", "TRANSACTION", "ONLINE", "PA", "SO", "TO", "BY", "VIA", "THE",

    # ── Payment Modes ──
    "ACH", "NACH", "ECS", "IMPS", "NEFT", "RTGS", "FT", "POS",

    # ── Corporate & Legal Suffixes (including truncated forms) ──
    "LIMITED", "LTD", "PRIVATE", "PVT", "BANK", "BANKING",
    "FINANCE", "FINSERV", "FINTECH",
    "TECHNOLOGIES", "TECH", "SOLUTIONS", "SERVICES",
    "INDIA", "CORP", "CORPORATION",
    "LIMITE", "LIMIT", "PRIVATEL", "PRIVAT",   # Truncated narration forms
    "DIGITAL", "COMMUNICATIONS", "COMM",
    "ENTERPRISES", "VENTURES", "ASSOCIATES",

    # ── Major Indian Banks & Their Acronyms ──
    "HDFC", "HDFCBANK",
    "ICICI", "ICICIBANK",
    "SBI", "SBIN", "STATE",
    "AXIS", "AXISBANK", "UTIB",
    "KOTAK", "KKBK",
    "YES", "YESB", "YESBANK", "YBS", "YESPAY",
    "INDUSIND", "INDUS", "IBL",
    "PNB", "PUNB", "PUNJAB", "NATIONAL",
    "BOB", "BARB", "BARODA",
    "IDFC", "IDFCBANK",
    "FEDERAL", "FDRL", "FBL",
    "RBL", "RATN",
    "CANARA", "CNRB",
    "UNION", "UBI", "UBIN",
    "CENTRAL", "CBIN",
    "INDIAN", "INDIANBK", "IDIB",
    "BOI", "BKID",
    "IOB", "IOBA",
    "KARB", "KARNATAKA",
    "SIBL",
    "BANDHAN", "BDBL",
    "IDBI", "IBKL",
    "UCO", "UCBA",
    "MAHB", "MAHARASHTRA",
    "PSIB",
    "EQUITAS", "UJJIVAN", "AUBANK",
    "FINO", "JIO", "NSDL",
    "CITI", "CITIBANK",
    "IPOS",

    # ── Payment Gateways & Fintechs ──
    "PHONEPE", "PAYTM", "GOOGLEPAY", "GPAY", "GOOGLE",
    "RAZORPAY", "BHARATPE", "AMAZONPAY", "RZP",
    "BILLDESK", "CCAVENUE", "PAYU", "CASHFREE", "PINELABS", "DIGIPE",
    "BHARTI", "AIRTEL", "BHARTIAIRTEL",
    "PLAYSTORE",

    # ── IMPS / NEFT Routing & Settlement Junk ──
    "ESCROW", "PAYOUT", "PAYOUTS", "STLMNT", "DOTRANSACTION",
    "AGGR", "AGG", "AGGREGATOR",
    "SETTL", "SETTLEMENT", "SETTLE",
    "NOD", "NODA", "NODAL",
    "ACCO", "MERCHANT",
    "RAPIPAY",
}


# ═════════════════════════════════════════════════════════════════════════════
# Phase 6 – Compound Word Splitting
#
# Indian bank narrations frequently concatenate multiple tokens into a single
# word with no spaces — particularly when gateway names, bank names, and
# corporate suffixes are merged together. These regex patterns are applied
# BEFORE tokenization in _clean_name. They insert spaces around recognized
# substrings so _strip_noise can individually identify and strip them.
#
# Examples:
#   "Phonepeprivatel"   → " Phonepe  privatel "   → both stripped → UNKNOWN
#   "Playstoreaxisbank" → " Playstore  axisbank " → both stripped → UNKNOWN
#   "Jiocitibank"       → "Jio citibank "         → both stripped → UNKNOWN
#   "Bhartiairtelltd"   → " Bharti  airtel  ltd " → all stripped  → UNKNOWN
# ═════════════════════════════════════════════════════════════════════════════

# Longer patterns first to avoid partial matches
_COMPOUND_GATEWAY_RE = re.compile(
    r"(phonepe|paytm|razorpay|bharatpe|amazonpay|googlepay|gpay"
    r"|sbiepay|billdesk|cashfree|pinelabs|playstore"
    r"|bhartiairtel|bharti|airtel|google|rapipay|digipe)",
    re.I,
)

_COMPOUND_BANK_RE = re.compile(
    r"(axisbank|hdfcbank|icicibank|indusind|citibank|yesbank"
    r"|kotakbank|canarabank|federalbank|idfcbank)",
    re.I,
)

# Truncated corporate suffixes glued to the previous word
_COMPOUND_SUFFIX_RE = re.compile(
    r"(privatel|private|limited|ltd|limite|limit|pvt)(?=[a-z]|$)",
    re.I,
)


# ═════════════════════════════════════════════════════════════════════════════
# Phase 7 (continued) – Pattern-Based Noise Removal
#
# Regex patterns catch technical artifacts that cannot be enumerated as
# fixed tokens: IFSC codes, bank routing codes, QR identifiers, IMPS
# routing codes (F09, F10), long alphanumeric reference IDs (10+ mixed
# characters), and masked mobile numbers (MOBILEXXXXX).
# ═════════════════════════════════════════════════════════════════════════════

BANK_OR_TECH_PATTERNS = (
    re.compile(r"^[A-Z]{4}0[A-Z0-9]{6}$"),           # IFSC codes (e.g., HDFC0001234)
    re.compile(r"^[A-Z]{3,6}\d{3,}$"),                # Bank routing codes (e.g., UTIB00512)
    re.compile(r"^PAYTMQR[A-Z0-9]*$", re.I),          # Paytm QR IDs
    re.compile(r"^Q\d+[A-Z0-9]*$", re.I),             # Generic QR IDs (e.g., Q899838377)
    re.compile(r"^[A-Z]{1,2}\d{5,}[A-Z0-9]*$", re.I), # QR/Reference prefixes (e.g., Zq7206840)
    re.compile(r"^P2[APM]$", re.I),                    # UPI transaction type flags
    re.compile(r"^\d{6,}$"),                           # Long numeric strings
    re.compile(r"^X{2,}\d*$", re.I),                   # Masked digits (XX, XXX...)
    re.compile(r"^XXXXX\d*$", re.I),                   # Masked account numbers
    re.compile(r"^MOBILEX+[A-Z0-9]*$", re.I),          # Masked mobile numbers
    re.compile(r"^(?=.*[A-Z])(?=.*\d)[A-Z0-9]{10,}$", re.I),  # Long alphanumeric reference IDs
    re.compile(r"^F\d{2}$", re.I),                     # IMPS routing codes (F09, F10, etc.)
)


# ═════════════════════════════════════════════════════════════════════════════
# Phase 7 (continued) – Standalone Bank Handle Stripping
#
# Bank handles that appear in narrations WITHOUT the @ prefix are erased
# so they don't get mistaken for counterparty names.
# ═════════════════════════════════════════════════════════════════════════════

_STANDALONE_HANDLE_RE = re.compile(
    r"\b(?:OKHDFC(?:BANK)?|OKICICI|OKSBI|OKAXIS"
    r"|YBL|IBL|AXL"
    r"|YESB|YESBANKLTD|YESBANK|SUPERYES|PTYES|YESCRED|YESPOP|YESPAY"
    r"|HDFCBANK|AXISBANK|IDFCBANK"
    r"|PHONEPEMERCHANT)\b",
    re.I,
)


# ═════════════════════════════════════════════════════════════════════════════
# Core Pipeline
#
# Entry point: parse_row() or parse_narration()
# Orchestrates all 9 phases in sequence for each transaction narration.
# ═════════════════════════════════════════════════════════════════════════════

def parse_row(row: Mapping[str, str]) -> ExtractionResult:
    return parse_narration(
        row.get("NARRATION", ""),
        transaction_type=row.get("TYPE", ""),
    )


def parse_narration(narration: str, transaction_type: str = "") -> ExtractionResult:
    raw = narration or ""

    # Phase 1 – Text Normalization
    normalized = _normalize(raw)

    # Phase 8 – Entity Role Classification
    flow = _flow_from_type(transaction_type)

    # Phase 2 – Payment Channel Detection
    payment_mode = _payment_mode(normalized)

    # Phases 3, 4, 5, 6, 7 – Counterparty Extraction, Bank Resolution,
    # Compound Word Splitting, and Noise Removal
    counterparty, confidence, rule = _counterparty(normalized, raw)

    # Phase 9 – Audit Metadata (confidence + rule packaged into result)
    return ExtractionResult(
        narration=raw,
        payment_mode=payment_mode,
        counterparty=counterparty,
        flow=flow,
        confidence=confidence,
        rule=rule,
    )


# ═════════════════════════════════════════════════════════════════════════════
# Phase 8 – Entity Role Classification
#
# The counterparty's role is inferred from the transaction type:
#   CREDIT → Sender (Payer / Money Source)
#   DEBIT  → Receiver (Payee / Money Recipient)
# ═════════════════════════════════════════════════════════════════════════════

def _flow_from_type(transaction_type: str) -> str:
    tx_type = (transaction_type or "").strip().upper()
    if tx_type == "DEBIT":
        return "Receiver"
    if tx_type == "CREDIT":
        return "Sender"
    return UNKNOWN


# ═════════════════════════════════════════════════════════════════════════════
# Phase 2 – Payment Channel Detection (implementation)
# ═════════════════════════════════════════════════════════════════════════════

def _payment_mode(text: str) -> str:
    # First check if UPI-based, then identify the specific gateway
    if re.search(r"\bUPI(?:OUT|AB|AR)?\b|UPI\s+IN|@[A-Z0-9._-]+", text, re.I):
        for gateway, pattern in GATEWAY_PATTERNS:
            if pattern.search(text):
                return gateway
        return "UPI"

    # If not UPI, classify into the appropriate banking channel
    for mode, pattern in MODE_PATTERNS:
        if pattern.search(text):
            return mode
    return UNKNOWN


# ═════════════════════════════════════════════════════════════════════════════
# Phases 3, 4, 5, 7 – Counterparty Extraction (Main Dispatcher)
#
# Orchestrates counterparty extraction across all channels:
#   1. Masked entity detection with bank resolution (Phase 5, 7)
#   2. UPI VPA extraction (Phase 3)
#   3. Channel-specific extractors for IMPS, NEFT, ACH, etc. (Phase 4)
#   4. Fallback cleaning and edge-case classification (Phase 7)
# ═════════════════════════════════════════════════════════════════════════════

def _counterparty(text: str, raw: str) -> tuple[str, float, str]:

    # ── Phase 7 – Masked Privacy Strings ─────────────────────────────────
    # Narrations containing heavily redacted identifiers (e.g., MOBILEXXXX)
    # are classified with contextual metadata: entity type (Merchant/
    # Individual) from P2M/P2P flags, and resolved bank name from the
    # UPI handle.
    if _is_masked_mobile(raw):
        suffix = ""
        if re.search(r"\bP2M\b", raw, re.I):
            suffix = " (Merchant)"
        elif re.search(r"\bP2P\b|\bP2A\b", raw, re.I):
            suffix = " (Individual)"

        # Phase 5 – Resolve the UPI handle to a bank name
        bank_name = _resolve_bank_from_handle(raw)
        if bank_name:
            suffix += f" - {bank_name}"

        return f"MASKED_COUNTERPARTY{suffix}", 0.99, "masked_mobile_rule"

    # ── Phase 3 – Counterparty Extraction (UPI) ─────────────────────────
    # Extracts the most meaningful counterparty from the Virtual Payment
    # Address (VPA). The portion before @ represents the merchant or
    # individual. QR-generated handles are rejected.
    if _is_upi(text):
        name = _extract_upi_counterparty(text)
        if name != UNKNOWN:
            return name, 0.86, "upi_vpa_or_name"

    # ── Phase 4 – Counterparty Extraction (Non-UPI) ─────────────────────
    # Format-specific extraction rules for IMPS, NEFT, IFN, ACH, POS/Card,
    # ATM, Cash, and Loan Recovery narrations.
    for extractor, confidence, rule in (
        (_extract_loan_counterparty,      0.88, "loan_recipient"),
        (_extract_ifn_counterparty,       0.82, "ifn_tail_name"),
        (_extract_imps_recd_counterparty, 0.82, "imps_recd_name"),
        (_extract_neft_counterparty,      0.80, "neft_named_segment"),
        (_extract_ach_counterparty,       0.78, "ach_originator"),
        (_extract_card_counterparty,      0.78, "card_merchant"),
        (_extract_cash_atm_counterparty,  0.72, "cash_atm_location"),
        (_extract_delimited_counterparty, 0.64, "best_delimited_segment"),
    ):
        candidate = extractor(text)
        if candidate != UNKNOWN:
            return candidate, confidence, rule

    # ── Phase 7 – Fallback: clean the raw narration ─────────────────────
    # Applies compound word splitting (Phase 6), noise removal (Phase 7),
    # and gibberish filtering before accepting the result.
    cleaned = _clean_name(raw)
    if cleaned and not _is_gibberish(cleaned):
        return cleaned, 0.45, "fallback_cleaned_narration"

    # ── Phase 7 – Informative Unknown Labels ────────────────────────────
    # Instead of assigning all unresolved cases to a generic UNKNOWN,
    # the parser classifies common edge cases into more informative
    # categories.

    # Unresolved QR Merchants
    if re.search(r"PAYTMQR|^(?:QR|Q)\d{5,}", raw, re.I):
        return "UNRESOLVED_QR_MERCHANT", 0.60, "qr_code_detected"

    # Transaction References (long mixed alphanumeric IDs)
    for part in _parts(text):
        if _is_reference_like(part):
            return TRANSACTION_REFERENCE, 0.40, "fallback_reference_id"

    # Unknown with Context (P2M/P2P with Phase 5 bank resolution)
    bank_name = _resolve_bank_from_handle(raw)
    bank_suffix = f" - {bank_name}" if bank_name else ""

    if re.search(r"\bP2M\b", raw, re.I):
        return f"UNKNOWN_MERCHANT (P2M){bank_suffix}", 0.40, "fallback_p2m"
    if re.search(r"\bP2P\b|\bP2A\b", raw, re.I):
        return f"UNKNOWN_INDIVIDUAL (P2P){bank_suffix}", 0.40, "fallback_p2p"

    return UNKNOWN, 0.0, "no_candidate"


# ═════════════════════════════════════════════════════════════════════════════
# Phase 1 – Text Normalization
#
# Transaction narrations are first normalized to improve parsing consistency.
# Common spacing inconsistencies (e.g., PH ONEPE, PHONE PE, PAY TM,
# GOOGLE PAY) and noisy delimiters (pipe | characters) are standardized so
# that payment gateway detection remains reliable across different banks
# and narration formats.
# ═════════════════════════════════════════════════════════════════════════════

def _normalize(text: str) -> str:
    text = text.replace("\ufeff", " ")
    text = re.sub(r"PH\s+ONE\s*PE|PHONE\s+PE", "PHONEPE", text, flags=re.I)
    text = re.sub(r"PAY\s+TM", "PAYTM", text, flags=re.I)
    text = re.sub(r"GOOGLE\s+PAY", "GOOGLEPAY", text, flags=re.I)
    text = re.sub(r"[|]+", "/", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def _is_upi(text: str) -> bool:
    return bool(re.search(r"\bUPI(?:OUT|AB|AR)?\b|UPI\s+IN|@[A-Z0-9._-]+", text, re.I))


# ═════════════════════════════════════════════════════════════════════════════
# Phase 3 & 4 – Channel-Specific Counterparty Extractors
#
# Phase 3 (UPI): Extracts counterparty from VPA handles.
# Phase 4 (Non-UPI): Format-specific rules for IMPS, NEFT, IFN, ACH,
#   POS/Card, ATM, Cash, and Loan Recovery narrations.
# ═════════════════════════════════════════════════════════════════════════════

def _extract_upi_counterparty(text: str) -> str:
    """Phase 3 – UPI: Extract counterparty from VPA (portion before @)."""
    parts = _parts(text)
    vpa_parts = [p for p in parts if "@" in p and not _is_masked_mobile(p)]

    for part in vpa_parts:
        before_at = part.split("@", 1)[0]
        candidate = _clean_vpa_handle(before_at)
        if _is_good_name(candidate):
            return candidate

    for part in parts:
        candidate = _clean_name(part)
        if _is_good_name(candidate):
            return candidate
    return UNKNOWN


def _extract_loan_counterparty(text: str) -> str:
    """Phase 4 – Loan Recovery: Extract recipient from LOAN REC narrations."""
    match = re.search(r"LOAN\s+REC[:/\s-]*(?:\d+\s*)?(?P<name>.+)$", text, re.I)
    return _clean_name(match.group("name")) if match else UNKNOWN


def _extract_ifn_counterparty(text: str) -> str:
    """Phase 4 – IFN: Extract name from the tail of IFN/ narrations."""
    match = re.search(r"\bIFN/[^ ]+\s+(?P<name>.+)$", text, re.I)
    return _clean_name(match.group("name")) if match else UNKNOWN


def _extract_imps_recd_counterparty(text: str) -> str:
    """Phase 4 – IMPS/RECD/MMT: Extract name from delimited segments."""
    parts = _parts(text)
    for marker in ("RECD", "IMPS", "MMT"):
        if parts and parts[0].upper().startswith(marker):
            for part in parts[2:]:
                candidate = _clean_name(part)
                if _is_good_name(candidate):
                    return candidate
    return UNKNOWN


def _extract_neft_counterparty(text: str) -> str:
    """Phase 4 – NEFT: Extract name from reversed segment scan."""
    parts = _parts(text)
    if not any(p.upper().startswith(("NEFT", "N")) for p in parts[:2]):
        return UNKNOWN
    for part in reversed(parts):
        candidate = _clean_name(part)
        if _is_good_name(candidate):
            return candidate
    return UNKNOWN


def _extract_ach_counterparty(text: str) -> str:
    """Phase 4 – ACH/NACH/ECS: Extract originator name."""
    parts = _parts(text)
    if not parts or not re.search(r"ACH|NACH|ECS", parts[0], re.I):
        return UNKNOWN
    for part in parts[1:]:
        candidate = _clean_name(part)
        if _is_good_name(candidate):
            return candidate
    return UNKNOWN


def _extract_card_counterparty(text: str) -> str:
    """Phase 4 – POS/Card: Extract merchant name."""
    parts = _parts(text)
    if not re.search(r"\b(?:POS|PRCR|CARD)\b", text, re.I):
        return UNKNOWN
    for part in parts[1:]:
        candidate = _clean_name(part)
        if _is_good_name(candidate):
            return candidate
    return UNKNOWN


def _extract_cash_atm_counterparty(text: str) -> str:
    """Phase 4 – ATM/Cash: Extract location or branch name."""
    if not re.search(r"\b(?:ATM|NFS|ATW|CASH)\b", text, re.I):
        return UNKNOWN
    parts = _parts(text)
    for part in reversed(parts):
        candidate = _clean_name(part)
        if _is_good_name(candidate):
            return candidate
    return UNKNOWN


def _extract_delimited_counterparty(text: str) -> str:
    """Phase 4 – Generic: Extract best candidate from delimited segments."""
    for part in _parts(text):
        candidate = _clean_name(part)
        if _is_good_name(candidate):
            return candidate
    return UNKNOWN


def _parts(text: str) -> list[str]:
    """Split narration text on / and # delimiters into segments."""
    return [part.strip() for part in re.split(r"[/#]", text) if part.strip()]


# ═════════════════════════════════════════════════════════════════════════════
# Phases 6 & 7 – Name Cleaning & Validation
#
# _clean_vpa_handle: Cleans UPI VPA handles (Phase 3 helper)
# _clean_name:       Master cleaning function applying Phase 6 (Compound
#                    Word Splitting) and Phase 7 (Noise Removal)
# _strip_noise:      Token-level noise word removal (Phase 7)
# _is_good_name:     Validates cleaned candidates against all Phase 7 rules
# ═════════════════════════════════════════════════════════════════════════════

def _clean_vpa_handle(handle: str) -> str:
    """
    Phase 3 helper – Cleans a UPI VPA handle (the portion before @).
    Strips QR prefixes, digits, and applies Phase 6 compound word splitting
    and Phase 7 noise removal.
    """
    compact = re.sub(r"[^A-Za-z0-9]", "", handle)
    if re.match(r"^PAYTMQR|^(?:QR|Q)\d", compact, re.I):
        return UNKNOWN
    if _digit_ratio(compact) > 0.45:
        return UNKNOWN
    handle = re.sub(r"\bMOBILEX+\b", "", handle, flags=re.I)
    handle = re.sub(r"^(?:paytmqr|qr|q)(?=[a-z0-9])", "", handle, flags=re.I)
    handle = re.sub(r"[-_.]+", " ", handle)
    handle = re.sub(r"\d+", " ", handle)

    # Phase 6 – Compound Word Splitting (applied to VPA handles too)
    handle = _COMPOUND_GATEWAY_RE.sub(r" \1 ", handle)
    handle = _COMPOUND_BANK_RE.sub(r" \1 ", handle)
    handle = _COMPOUND_SUFFIX_RE.sub(r" \1 ", handle)

    # Phase 7 – Token-level noise removal
    return _title(_strip_noise(handle))


def _clean_name(value: str) -> str:
    """
    Master cleaning function for counterparty name candidates.
    Applies Phases 6 (Compound Word Splitting) and 7 (Noise Removal).
    """
    # Strip @ handles, masked fields, dates, times, long numbers
    value = re.sub(r"@[A-Z0-9._-]+", " ", value, flags=re.I)
    value = re.sub(r"\bMOBILEX+\b", " ", value, flags=re.I)
    value = re.sub(r"\bX{4,}\d*\b", " ", value, flags=re.I)
    value = re.sub(r"\b\d{2}[-/]\d{2}[-/]\d{2,4}\b", " ", value)
    value = re.sub(r"\b\d{1,2}:\d{2}(?::\d{2})?\b", " ", value)
    value = re.sub(r"\b\d{4,}\b", " ", value)

    # Phase 7 – Erase standalone bank handles missing the @ symbol
    value = _STANDALONE_HANDLE_RE.sub(" ", value)

    # Phase 6 – Compound Word Splitting
    # Insert spaces around known gateway/bank/suffix substrings so that
    # _strip_noise can see them as individual tokens and remove them.
    value = _COMPOUND_GATEWAY_RE.sub(r" \1 ", value)
    value = _COMPOUND_BANK_RE.sub(r" \1 ", value)
    value = _COMPOUND_SUFFIX_RE.sub(r" \1 ", value)

    value = re.sub(r"[-_.]+", " ", value)

    # Phase 7 – Token-level noise removal
    value = _strip_noise(value)
    return _title(value)


def _strip_noise(value: str) -> str:
    """
    Phase 7 – Token-level noise removal.
    Iterates through each alphabetic token and removes it if it matches
    a known noise word, a technical/routing pattern, or is a single character.
    """
    tokens = []
    for token in re.findall(r"[A-Za-z][A-Za-z0-9&]*", value):
        upper = token.upper()
        if upper in NOISE_WORDS:
            continue
        if any(pattern.match(upper) for pattern in BANK_OR_TECH_PATTERNS):
            continue
        if len(token) == 1 and upper not in {"D"}:
            continue
        tokens.append(token)
    return " ".join(tokens).strip()


def _title(value: str) -> str:
    """Title-case the cleaned name. Short all-caps words (≤4 chars) are kept as-is."""
    if not value:
        return UNKNOWN
    words = []
    for word in value.split():
        if word.isupper() and len(word) <= 4:
            words.append(word)
        else:
            words.append(word[:1].upper() + word[1:].lower())
    return " ".join(words)


def _is_good_name(value: str) -> bool:
    """
    Phase 7 – Validates whether a cleaned candidate is good enough to keep
    as a counterparty. Prevents banking labels, QR IDs, reference IDs, and
    gibberish from being treated as merchants or people.
    """
    if not value or value == UNKNOWN:
        return False

    # Allow informative tags to pass through
    if value.startswith(("UNKNOWN_", "UNRESOLVED_", "MASKED_")):
        return True

    if len(value) < 3:
        return False

    upper = value.upper().replace(" ", "")

    if upper in NOISE_WORDS:
        return False

    if any(pattern.match(upper) for pattern in BANK_OR_TECH_PATTERNS):
        return False

    # Phase 7 – Transaction Reference detection
    if _is_reference_like(value):
        return False

    # Phase 7 – Gibberish Filtering
    if _is_gibberish(value):
        return False

    return bool(re.search(r"[A-Za-z]", value))


# ═════════════════════════════════════════════════════════════════════════════
# Phase 7 – Utility Helpers (Masking, References, Gibberish)
# ═════════════════════════════════════════════════════════════════════════════

def _is_masked_mobile(value: str) -> bool:
    """Phase 7 – Detects masked mobile numbers (e.g., MOBILEXXXX)."""
    return bool(re.search(r"MOBILEX+", value, re.I))


def _digit_ratio(value: str) -> float:
    """Returns the ratio of digit characters to total characters."""
    if not value:
        return 0.0
    digits = sum(c.isdigit() for c in value)
    return digits / len(value)


def _is_reference_like(value: str) -> bool:
    """
    Phase 7 – Transaction Reference detection.
    Returns True when the extracted value looks like a system/reference ID
    instead of a real person or merchant name. Long mixed alphanumeric
    strings (10+ characters) are classified as references.

    Bk9ew562zbcpvu252290238983  →  True   (reference ID)
    Redeepesh                   →  False  (real name)
    """
    value = re.sub(r"[^A-Za-z0-9]", "", str(value).strip())
    if not value:
        return False
    has_letter = any(c.isalpha() for c in value)
    has_digit = any(c.isdigit() for c in value)
    if has_letter and has_digit and len(value) >= 10:
        return True
    return False


def _is_gibberish(value: str) -> bool:
    """
    Phase 7 – Gibberish Filtering.
    Checks if a string is a random cluster of consonants. Real Indian names
    and merchants contain at least one vowel or 'Y'. Random character
    sequences (e.g., Zxzy, Zvm) are detected and rejected.
    """
    if value.upper() in {"P2A", "P2P", "P2M"}:
        return False
    if not re.search(r"[AEIOUYaeiouy]", value):
        return True
    return False


In [ ]:
'''# Search for any extracted counterparty that still sounds like a bank or company
mask = df['COUNTERPARTY'].str.contains(r'Bank|Ltd|Pvt|Private|Limited|Fintech', case=False, na=False)

# Display the top 50 offenders
print(df[mask]['COUNTERPARTY'].value_counts().head(50))'''

COUNTERPARTY
MASKED_COUNTERPARTY - YES Bank                        16231
MASKED_COUNTERPARTY - Axis Bank                        9618
MASKED_COUNTERPARTY - IndusInd Bank                    6557
Jiocitibank                                            2100
Phonepeprivatel                                         344
MASKED_COUNTERPARTY - State Bank of India               205
Bhartiairtelltd Rzp                                     184
MASKED_COUNTERPARTY - ICICI Bank                        158
MASKED_COUNTERPARTY - Paytm Payments Bank               154
Playstoreaxisbank                                       153
MASKED_COUNTERPARTY - Canara Bank                       131
NKPR Fintech SO                                          99
MASKED_COUNTERPARTY - Airtel Payments Bank               89
MASKED_COUNTERPARTY - HDFC Bank                          83
MASKED_COUNTERPARTY - RBL Bank                           69
Phonepelimitedp                                          68
MASKED_COUNTERPARTY - Kotak

In [ ]:
'''# Find counterparties that still look like bank names
mask = df['COUNTERPARTY'].str.contains(
    r'Bank|Ltd|Pvt|Private|Limited|Fintech|Escrow|Payout',
    case=False, na=False
)
print(df[mask]['COUNTERPARTY'].value_counts().head(30))'''

COUNTERPARTY
MASKED_COUNTERPARTY - YES Bank                                         8692
MASKED_COUNTERPARTY - IndusInd Bank                                    4634
MASKED_COUNTERPARTY - Axis Bank                                        4193
Jiocitibank                                                            2100
YES BANK Limit                                                         1083
Yes Bank Ltd                                                            957
YES BANK Limite                                                         866
Phonepe Limited AGGR                                                    453
Phonepe Private Limited                                                 382
Phonepe Limited F09                                                     349
Phonepeprivatel                                                         344
Google India Digital Services Private Limited NOD Dharmendra Singh      312
ONE Communications Limited Settl                                        306

In [ ]:
# Run the parser (No external `.apply()` cleaner needed anymore!)
parsed = df.apply(lambda row: parse_row(row).to_dict(), axis=1)
parsed_df = pd.DataFrame(parsed.tolist())

# Map the columns
df["PAYMENT_APP"] = parsed_df["payment_mode"]
df["COUNTERPARTY"] = parsed_df["counterparty"]
df["ENTITY_ROLE"] = parsed_df["flow"].map({
    "Sender": "Payer (Source)",
    "Receiver": "Payee (Recipient)"
}).fillna("Unknown Role")
df["EXTRACTION_CONFIDENCE"] = parsed_df["confidence"]
df["EXTRACTION_RULE"] = parsed_df["rule"]

df.head()

,NARRATION,AAID,TYPE,MODE,AMOUNT,CURRENTBALANCE,VALUEDATE,PAYMENT_APP,COUNTERPARTY,ENTITY_ROLE,EXTRACTION_CONFIDENCE,EXTRACTION_RULE
0,UPIOUT/415442131843/Q899838377@ybl/Payment f/5411,aa91777914703824768192900488347968430422,DEBIT,OTHERS,70,7530,00:00.0,PhonePe,UNKNOWN,Payee (Recipient),0.45,fallback_cleaned_narration
1,UPIOUT/415958720613/paytmqre4lk0uevrg@paytm//5451,aa91777914703824768192900488347968430422,DEBIT,OTHERS,60,1649,00:00.0,Paytm,UNKNOWN,Payee (Recipient),0.45,fallback_cleaned_narration
2,UPIOUT/416160374009/paytmqr14clec@paytm/Paym/5812,aa91777914703824768192900488347968430422,DEBIT,OTHERS,36,8730,00:00.0,Paytm,Qr14clec,Payee (Recipient),0.86,upi_vpa_or_name
3,UPIOUT/415838737884/Q047024853@ybl/Payment f/5812,aa91777914703824768192900488347968430422,DEBIT,OTHERS,5,8019,00:00.0,PhonePe,UNKNOWN,Payee (Recipient),0.45,fallback_cleaned_narration
4,UPIOUT/416102104755/paytmqr281005050101wntva/5411,aa91777914703824768192900488347968430422,DEBIT,OTHERS,24,11040,00:00.0,Paytm,UNKNOWN,Payee (Recipient),0.45,fallback_cleaned_narration


In [ ]:
df[['PAYMENT_APP', 'COUNTERPARTY']].value_counts().reset_index()

,PAYMENT_APP,COUNTERPARTY,count
0,UPI,MASKED_COUNTERPARTY,50109
1,UPI,UNKNOWN,31628
2,PhonePe,MASKED_COUNTERPARTY - YES Bank,15448
3,PhonePe,MASKED_COUNTERPARTY - Axis Bank,9474
4,Paytm,UNKNOWN,6653
...,...,...,...
42068,UPI,Harshitsingh,1
42069,UPI,Harshitsaxena,1
42070,UPI,Harshitkumarsin,1
42071,UPI,Harshitha KR,1
